# UrbanFloodBench Baseline Workflow

This notebook is the team-friendly entry point for:

- loading UrbanFloodBench data
- building cleaned node-level features
- running persistence and random forest baselines
- inspecting validation metrics

The notebook calls reusable code from `src/urbanflood`, so the team can collaborate in notebooks without duplicating logic.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

WindowsPath('D:/Desktop/lood')

In [3]:
import pandas as pd

from urbanflood.dataset import load_model_assets
from urbanflood.features import build_training_dataset, clean_training_dataset
from urbanflood.baselines import (
    event_based_split,
    get_feature_columns,
    run_persistence_baseline,
    run_random_forest_baseline,
    evaluate_baselines,
)

pd.set_option('display.max_columns', 200)

## Paths And Debug Settings

In [4]:
DATA_DIR = ROOT / 'Data'
MAX_EVENTS_PER_MODEL = 2
MAX_TRAIN_ROWS = 50000
LAG_STEPS = (1, 2, 3, 5, 10)
RAIN_WINDOWS = (3, 5, 10)

DATA_DIR

WindowsPath('D:/Desktop/lood/Data')

## Build Training Dataset

In [5]:
frames = []
for model_id in (1, 2):
    assets = load_model_assets(DATA_DIR, model_id=model_id, split='train')
    frame = build_training_dataset(
        assets,
        lag_steps=LAG_STEPS,
        rain_windows=RAIN_WINDOWS,
        max_events=MAX_EVENTS_PER_MODEL,
    )
    frame = clean_training_dataset(frame)
    print(f'model {model_id}: {len(frame):,} rows')
    frames.append(frame)

dataset = pd.concat(frames, ignore_index=True)
dataset.shape

model 1: 1,108,701 rows
model 2: 1,349,100 rows


(2457801, 83)

In [6]:
dataset.head()

,timestep,node_idx,water_level,node_aux_flow,model_id,event_id,node_type,position_x,position_y,depth,invert_elevation,surface_elevation,base_area,edge_in_flow_mean,edge_in_flow_max,edge_in_flow_min,edge_in_velocity_mean,edge_in_velocity_max,edge_in_velocity_min,edge_out_flow_mean,edge_out_flow_max,edge_out_flow_min,edge_out_velocity_mean,edge_out_velocity_max,edge_out_velocity_min,edge_upstream_water_level_mean,edge_upstream_water_level_max,edge_upstream_water_level_min,edge_downstream_water_level_mean,edge_downstream_water_level_max,edge_downstream_water_level_min,surface_paired_water_level_mean,surface_paired_water_level_max,surface_paired_water_level_min,surface_paired_rainfall_mean,surface_paired_rainfall_max,surface_paired_rainfall_min,surface_paired_water_volume_mean,surface_paired_water_volume_max,surface_paired_water_volume_min,effective_area,reference_elevation,rainfall,target_water_level,target_delta_water_level,delta_water_level,water_level_lag_1,delta_water_level_lag_1,rainfall_lag_1,water_level_lag_2,delta_water_level_lag_2,rainfall_lag_2,water_level_lag_3,delta_water_level_lag_3,rainfall_lag_3,water_level_lag_5,delta_water_level_lag_5,rainfall_lag_5,water_level_lag_10,delta_water_level_lag_10,rainfall_lag_10,rainfall_sum_3,rainfall_mean_3,rainfall_sum_5,rainfall_mean_5,rainfall_sum_10,rainfall_mean_10,cumulative_rainfall,timestep_index,water_level_above_reference,is_raining,water_level_to_area_ratio,water_volume,area,roughness,min_elevation,elevation,aspect,curvature,flow_accumulation,pipe_paired_water_level_mean,pipe_paired_water_level_max,pipe_paired_water_level_min
0,0.0,0,294.87430,0.0,1,1,1,802465.6,349898.84,8.977997,292.342,301.32,12.56,16.837931,31.815710,1.860152,4.970807,6.481443,3.460171,33.611320,33.611320,33.611320,8.418276,8.418276,8.418276,296.187460,296.18870,296.18622,288.22095,288.22095,288.22095,300.070648,300.070648,300.070648,0.143333,0.143333,0.143333,1.180193,1.180193,1.180193,12.56,292.342,0.143333,294.98570,0.11140,0.00000,0.00000,0.00000,0.000000,0.00000,0.00000,0.000000,0.0000,0.0000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.143333,0.143333,0.143333,0.143333,0.143333,0.143333,0.143333,0,2.53230,1,0.201616,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0,294.98570,0.0,1,1,1,802465.6,349898.84,8.977997,292.342,301.32,12.56,17.313536,32.773335,1.853738,5.056079,6.676529,3.435629,34.621025,34.621025,34.621025,8.554404,8.554404,8.554404,296.282965,296.37643,296.18950,288.25427,288.25427,288.25427,300.070374,300.070374,300.070374,0.136667,0.136667,0.136667,1.178370,1.178370,1.178370,12.56,292.342,0.136667,295.00534,0.01964,0.11140,294.87430,0.00000,0.143333,0.00000,0.00000,0.000000,0.0000,0.0000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.280000,0.140000,0.280000,0.140000,0.280000,0.140000,0.280000,1,2.64370,1,0.210486,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2.0,0,295.00534,0.0,1,1,1,802465.6,349898.84,8.977997,292.342,301.32,12.56,17.397679,32.941055,1.854302,5.071252,6.710697,3.431807,34.790653,34.790653,34.790653,8.577585,8.577585,8.577585,296.300165,296.41030,296.19003,288.25964,288.25964,288.25964,300.070435,300.070435,300.070435,0.136667,0.136667,0.136667,1.178905,1.178905,1.178905,12.56,292.342,0.136667,295.02298,0.01764,0.01964,294.98570,0.11140,0.136667,294.87430,0.00000,0.143333,0.0000,0.0000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.416667,0.138889,0.416667,0.138889,0.416667,0.138889,0.416667,2,2.66334,1,0.212049,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3.0,0,295.02298,0.0,1,1,1,802465.6,349898.84,8.977997,292.342,301.32,12.56,17.553244,33.064080,2.042407,5.138633,6.735759,3.541506,35.096320,35.096320,35.096320,8.619795,8.619795,8.619795,296.327085,296.43850,296.21567,288.26910,288.26910,288.26910,300.070404,300.070404,300.070404,0.136667,0.136667,0.136667,1.178556,1.178556,1.178556,12.56,292.342,0.136667,295.02390,0.00092,0.01764,295.00534,0.01964,0.136667,294.98570,0.11140,0.136667,294.8743,0.0000,0.143333,0.0,0.0,0.0,0.0,0.0,0.0,0.410000,0.136667,0.553333,0.138333,0.553333,0.138333,0.553

## Event-Based Validation Split

In [7]:
train_df, val_df = event_based_split(dataset, validation_fraction=0.2)
if len(train_df) > MAX_TRAIN_ROWS:
    train_df = train_df.sample(MAX_TRAIN_ROWS, random_state=42)

feature_columns = get_feature_columns(
    train_df,
    drop_columns=('target_water_level', 'target_delta_water_level', 'timestamp'),
)

train_df.shape, val_df.shape, len(feature_columns)

((50000, 83), (1678920, 83), 76)

## Run Baselines

In [8]:
persistence_pred = run_persistence_baseline(val_df)

rf_pred, rf_model = run_random_forest_baseline(
    train_df=train_df,
    val_df=val_df,
    feature_columns=feature_columns,
    n_estimators=300,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

metrics = evaluate_baselines([persistence_pred, rf_pred])
metrics

,model_name,rmse,mae,num_rows
0,persistence,0.022982,0.008444,1678920
1,random_forest,0.025243,0.009622,1678920


## Inspect Predictions

In [9]:
rf_pred.head()

,model_id,event_id,node_type,node_idx,timestep,target_water_level,prediction,model_name
0,1,2,1,0,0.0,292.80490,291.664905,random_forest
1,1,2,1,0,1.0,292.78700,292.275850,random_forest
2,1,2,1,0,2.0,292.77158,292.521654,random_forest
3,1,2,1,0,3.0,292.76025,292.490809,random_forest
4,1,2,1,0,4.0,292.75177,292.419370,random_forest
